# Tratamento dos Dados — População por Faixa Etária



---
## 1. Imports

In [1]:
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais

Centraliza todos os parâmetros configuráveis. Ajuste aqui antes de executar o pipeline.

In [2]:

# ── Caminhos ─────────────────────────────────────────────────────────────────
ENTRADA    = Path("/home/anna/tcc/Limpeza e Integração/População - Faixa Etária")
SAIDA      = Path("/home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv")
MUNICIPIOS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

# ── Colunas-chave presentes nos CSVs de origem ───────────────────────────────
# Usadas para identificar e unir os dois arquivos de cada ano (parte 1 e parte 2)
CHAVES = ["Sigla", "Código", "Município"]

# ── Faixas etárias de entrada (como aparecem nos CSVs do IBGE) ───────────────
FAIXAS = [
    "0 a 9",
    "10 a 19",
    "20 a 29",
    "30 a 39",
    "40 a 49",
    "50 a 59",
    "60 a 69",
    "70 anos ou mais",
]

# ── Mapeamento entrada → saída (renomeia faixas para o padrão do TCC) ─────────
FAIXA_SAIDA = {
    "0 a 9"          : "0-9",
    "10 a 19"        : "10-19",
    "20 a 29"        : "20-29",
    "30 a 39"        : "30-39",
    "40 a 49"        : "40-49",
    "50 a 59"        : "50-59",
    "60 a 69"        : "60-69",
    "70 anos ou mais": "70 OU MAIS",
}

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")
print(f"   Faixas configuradas: {list(FAIXA_SAIDA.values())}")


✅ Pasta de saída: /home/anna/tcc/Seleção e Transformação/População
   Faixas configuradas: ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70 OU MAIS']


---
## 3. Funções de Normalização de Texto

Funções para padronizar nomes de municípios antes do join com a tabela de códigos IBGE.

In [3]:
def normalizar_texto(texto: str) -> str:
    """Remove acentos e converte para minúsculas. Trata NaN como string vazia."""
    texto = "" if pd.isna(texto) else str(texto).strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower()


def normalizar_chave_municipio(texto: str) -> str:
    """
    Normalização completa para uso como chave de join:
    - Aplica normalizar_texto
    - Substitui hífens e apóstrofos por espaço
    - Colapsa espaços múltiplos
    - Resolve aliases de grafias conhecidas
    """
    chave = normalizar_texto(texto)
    chave = chave.replace("-", " ").replace("'", " ")
    chave = re.sub(r"\s+", " ", chave).strip()
    aliases = {
        "dona eusebia"          : "dona euzebia",
        "sao thome das letras"  : "sao tome das letras",
        "brasopolis"            : "brazopolis",
        "itabirinha de mantena" : "itabirinha",
    }
    return aliases.get(chave, chave)


print("✅ Funções de normalização definidas.")

✅ Funções de normalização definidas.


---
## 4. Funções de Leitura dos Arquivos

Os CSVs do IBGE usam `sep=";"` e `encoding="utf-8"`. Cada ano vem em dois arquivos:
- **Parte 1** (`1-YYYY.csv`): primeiras faixas etárias
- **Parte 2** (`2-YYYY.csv`): demais faixas etárias

`extrair_ano_e_parte` valida o nome do arquivo e extrai ano e número da parte.

In [4]:
def limpar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Remove espaços dos nomes de colunas e descarta colunas 'Unnamed:...'."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df.loc[:, ~df.columns.str.match(r"^Unnamed")].copy()


def ler_arquivo_csv(caminho: Path) -> pd.DataFrame:
    """Lê um CSV do IBGE (sep=';', utf-8) e limpa as colunas."""
    return limpar_colunas(pd.read_csv(caminho, sep=";", encoding="utf-8"))


def extrair_ano_e_parte(caminho: Path) -> tuple[int, int]:
    """
    Extrai (ano, parte) do nome do arquivo no padrão '1-YYYY.csv' ou '2-YYYY.csv'.
    Lança ValueError se o nome não corresponder ao padrão esperado.
    """
    match = re.fullmatch(r"([12])-(\d{4})\.csv", caminho.name)
    if not match:
        raise ValueError(f"Nome de arquivo inesperado: {caminho.name}")
    return int(match.group(2)), int(match.group(1))


print("✅ Funções de leitura definidas.")

✅ Funções de leitura definidas.


---
## 5. Funções de Construção da Tabela por Ano

`faixa_decenal` detecta a qual faixa padronizada pertence cada coluna do CSV (ex: "pessoas 40 a 44 anos" → "40 a 49").  
`montar_tabela_ano` une as duas partes do ano, agrega as colunas nas 8 faixas e renomeia para `YYYY_faixa`.

In [5]:
def faixa_decenal(nome_coluna: str) -> str | None:
    """
    Mapeia o nome de uma coluna do CSV para a faixa decenal padronizada.
    Exemplos:
      'pessoas 40 a 44 anos' → '40 a 49'
      'pessoas 70 anos ou mais' → '70 anos ou mais'
    Retorna None se a coluna não corresponder a nenhuma faixa.
    """
    intervalo = re.search(r"pessoas (\d+) a (\d+) anos", nome_coluna)
    if intervalo:
        inicio_decada = (int(intervalo.group(1)) // 10) * 10
        return f"{inicio_decada} a {inicio_decada + 9}"

    if re.search(r"pessoas 70 anos ou mais", nome_coluna):
        return "70 anos ou mais"

    return None


def nome_coluna_saida(ano: int, faixa: str) -> str:
    """Monta o nome de coluna de saída no padrão YYYY_faixa (ex: '2010_40-49')."""
    return f"{ano}_{FAIXA_SAIDA[faixa]}"


def montar_tabela_ano(ano: int, arquivos_do_ano: dict[int, Path]) -> pd.DataFrame:
    """
    Une os dois arquivos do ano (parte 1 e parte 2), agrega colunas nas 8 faixas
    padronizadas e renomeia para o formato YYYY_faixa.
    """
    faltando = {1, 2} - set(arquivos_do_ano)
    if faltando:
        raise ValueError(f"Faltam arquivos do ano {ano}: partes {sorted(faltando)}")

    df_1 = ler_arquivo_csv(arquivos_do_ano[1])
    df_2 = ler_arquivo_csv(arquivos_do_ano[2])

    # Une as duas partes mantendo apenas colunas novas da parte 2
    colunas_parte2 = [c for c in df_2.columns if c not in CHAVES]
    combinado = df_1.merge(df_2[CHAVES + colunas_parte2], on=CHAVES, how="inner")

    # Inicia resultado com as colunas-chave e faixas zeradas
    resultado = combinado[CHAVES].copy()
    for faixa in FAIXAS:
        resultado[faixa] = 0.0

    # Agrega cada coluna de população na sua faixa decenal
    for col in [c for c in combinado.columns if c not in CHAVES]:
        faixa = faixa_decenal(col)
        if faixa is not None:
            resultado[faixa] += pd.to_numeric(combinado[col], errors="coerce").fillna(0)

    # Renomeia faixas para o padrão YYYY_faixa
    return resultado.rename(columns={f: nome_coluna_saida(ano, f) for f in FAIXAS})


print("✅ Funções de construção da tabela por ano definidas.")

✅ Funções de construção da tabela por ano definidas.


---
## 6. Enriquecimento com Código IBGE

Carrega o mapeamento `municipio_normalizado → codigo_ibge` e faz o join pelo nome normalizado.

In [6]:
def carregar_mapa_codigos(arquivo: Path = MUNICIPIOS) -> dict[str, str]:
    """
    Carrega o mapeamento municipio_normalizado → codigo_ibge.
    Lança FileNotFoundError ou ValueError se o arquivo estiver ausente ou incompleto.
    """
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo de municípios não encontrado: {arquivo}")

    df_ref = pd.read_csv(arquivo, dtype={"CÓDIGO IBGE": "string"})
    df_ref = df_ref.rename(columns={"CÓDIGO IBGE": "codigo_ibge", "MUNICÍPIO NORMALIZADO": "municipio_normalizado"})
    obrigatorias = {"municipio_normalizado", "codigo_ibge"}
    faltantes    = obrigatorias - set(df_ref.columns)
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes: {sorted(faltantes)}")

    df_ref = df_ref.dropna(subset=["municipio_normalizado", "codigo_ibge"]).copy()
    df_ref["municipio_normalizado"] = df_ref["municipio_normalizado"].astype(str).map(normalizar_chave_municipio)
    df_ref["codigo_ibge"]           = df_ref["codigo_ibge"].astype(str).str.strip()
    df_ref = df_ref.drop_duplicates(subset=["municipio_normalizado"], keep="first")

    return dict(zip(df_ref["municipio_normalizado"], df_ref["codigo_ibge"]))


def adicionar_coluna_codigo(df: pd.DataFrame, mapa_codigos: dict[str, str]) -> pd.DataFrame:
    """
    Insere a coluna 'CÓDIGO' logo após 'Município' via join pelo nome normalizado.
    Lista municípios sem correspondência para revisão manual.
    """
    if "Município" not in df.columns:
        raise ValueError("DataFrame não possui a coluna obrigatória 'Município'.")

    chaves  = df["Município"].astype(str).map(normalizar_chave_municipio)
    codigos = chaves.map(mapa_codigos)

    sem_cod = df.loc[codigos.isna(), "Município"].astype(str).dropna().sort_values().unique().tolist()
    if sem_cod:
        print(f"⚠️  {len(sem_cod)} município(s) sem código IBGE:")
        for m in sem_cod:
            print(f"   - {m}")
    else:
        print("✅ Todos os municípios tiveram código IBGE encontrado.")

    df = df.drop(columns=["CÓDIGO"], errors="ignore").copy()
    pos = list(df.columns).index("Município") + 1
    df.insert(pos, "CÓDIGO", codigos)
    return df


print("✅ Funções de código IBGE definidas.")

✅ Funções de código IBGE definidas.


---
## 7. Pipeline de Tratamento

Executa o pipeline completo em 4 etapas:
1. Localiza e agrupa os arquivos por ano
2. Constrói a tabela de cada ano e une tudo
3. Calcula totais por ano (`TOTAL_YYYY`) e ordena colunas
4. Adiciona código IBGE e salva o CSV final

In [7]:
# ── Etapa 1: Localizar e agrupar arquivos por ano ────────────────────────────
print("🔍 Localizando arquivos...")
arquivos = sorted(ENTRADA.glob("*.csv"))
if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {ENTRADA}")

arquivos_por_ano: dict[int, dict[int, Path]] = {}
for arq in arquivos:
    try:
        ano, parte = extrair_ano_e_parte(arq)
        arquivos_por_ano.setdefault(ano, {})[parte] = arq
    except ValueError as e:
        print(f"   ⚠️  Ignorado: {e}")

anos_ordenados = sorted(arquivos_por_ano)
print(f"   Anos encontrados: {anos_ordenados}")
for ano in anos_ordenados:
    partes = sorted(arquivos_por_ano[ano].keys())
    print(f"   {ano}: partes {partes}")

🔍 Localizando arquivos...
   Anos encontrados: [1970, 1980, 1991, 2000, 2010, 2022]
   1970: partes [1, 2]
   1980: partes [1, 2]
   1991: partes [1, 2]
   2000: partes [1, 2]
   2010: partes [1, 2]
   2022: partes [1, 2]


In [8]:
# ── Etapa 2: Construir tabelas por ano e unir ────────────────────────────────
print("\n⚙️  Construindo tabelas por ano...")
tabelas_por_ano = []
for ano in anos_ordenados:
    t = montar_tabela_ano(ano, arquivos_por_ano[ano])
    print(f"   {ano}: {len(t)} municípios, {len(t.columns)} colunas")
    tabelas_por_ano.append(t)

tabela_final = tabelas_por_ano[0].copy()
for tabela_ano in tabelas_por_ano[1:]:
    colunas_novas = [c for c in tabela_ano.columns if c not in CHAVES]
    tabela_final  = tabela_final.merge(
        tabela_ano[CHAVES + colunas_novas], on=CHAVES, how="outer"
    )

# Ordena colunas: CHAVES + faixas por ano em ordem
ordem = CHAVES + [nome_coluna_saida(ano, faixa) for ano in anos_ordenados for faixa in FAIXAS]
tabela_final = tabela_final[ordem].sort_values("Município").reset_index(drop=True)

# Converte colunas numéricas para int (fillna 0)
cols_num = [c for c in tabela_final.columns if c not in CHAVES]
tabela_final[cols_num] = tabela_final[cols_num].fillna(0).astype(int)

print(f"\n✅ Tabela base: {tabela_final.shape}")


⚙️  Construindo tabelas por ano...
   1970: 853 municípios, 11 colunas
   1980: 853 municípios, 11 colunas
   1991: 853 municípios, 11 colunas
   2000: 853 municípios, 11 colunas
   2010: 853 municípios, 11 colunas
   2022: 853 municípios, 11 colunas

✅ Tabela base: (853, 51)


In [9]:
# ── Etapa 3: Calcular totais por ano ─────────────────────────────────────────
print("\n⚙️  Calculando totais por ano...")
for ano in anos_ordenados:
    cols_faixas        = [nome_coluna_saida(ano, f) for f in FAIXAS]
    tabela_final[f"TOTAL_{ano}"] = tabela_final[cols_faixas].sum(axis=1).astype(int)

# Reordena: CHAVES + (faixas + total por ano) interleaved
faixas_todas  = [nome_coluna_saida(ano, f) for ano in anos_ordenados for f in FAIXAS]
totais_ano    = [f"TOTAL_{ano}" for ano in anos_ordenados]
tabela_final  = tabela_final[list(CHAVES) + faixas_todas + totais_ano]

print(f"✅ Shape após totais: {tabela_final.shape}")


⚙️  Calculando totais por ano...
✅ Shape após totais: (853, 57)


In [10]:

# ── Etapa 4: Código IBGE, Regional de Saúde, renomear e salvar ───────────────
print("\n⚙️  Adicionando código IBGE...")

# Remove colunas de origem (Sigla e Código do IBGE; será substituído pelo IBGE normalizado)
tabela_final = tabela_final.drop(columns=["Sigla", "Código"])
tabela_final["Município"] = tabela_final["Município"].map(normalizar_texto)

mapa_codigos = carregar_mapa_codigos()
tabela_final = adicionar_coluna_codigo(tabela_final, mapa_codigos)

# Limpa o código IBGE (remove '.0' residual) e padroniza nomes de colunas
if "CÓDIGO" in tabela_final.columns:
    tabela_final["CÓDIGO"] = tabela_final["CÓDIGO"].astype(str).str.replace(r"\.0$", "", regex=True)
    tabela_final = tabela_final.rename(columns={"CÓDIGO": "CÓDIGO IBGE"})

tabela_final = tabela_final.rename(columns={"Município": "MUNICÍPIO"})

# ── Regional de Saúde, via join pelo código IBGE completo ───────────────────
print("⚙️  Adicionando Regional de Saúde...")
df_mun = pd.read_csv(MUNICIPIOS, dtype=str)
mapa_regional = df_mun.set_index("CÓDIGO IBGE")["REGIONAL NORMALIZADA"].to_dict()
tabela_final["REGIONAL DE SAÚDE"] = tabela_final["CÓDIGO IBGE"].map(mapa_regional)

sem_regional = int(tabela_final["REGIONAL DE SAÚDE"].isna().sum())
if sem_regional > 0:
    print(f"⚠️  {sem_regional} município(s) sem Regional de Saúde correspondente.")
else:
    print("✅ Todos os municípios tiveram Regional de Saúde encontrada.")

# Renomeia TOTAL_YYYY → YYYY_TOTAL (padrão do TCC)
tabela_final = tabela_final.rename(columns={f"TOTAL_{ano}": f"{ano}_TOTAL" for ano in anos_ordenados})

# Reordena: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, faixas por ano, totais por ano
faixas_todas = [nome_coluna_saida(ano, f) for ano in anos_ordenados for f in FAIXAS]
totais_ano   = [f"{ano}_TOTAL" for ano in anos_ordenados]
tabela_final = tabela_final[["MUNICÍPIO", "CÓDIGO IBGE", "REGIONAL DE SAÚDE"] + faixas_todas + totais_ano]

# Salva
tabela_final.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("\n╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo  : {str(SAIDA.name):<41}║")
print(f"║  Linhas   : {len(tabela_final):<41}║")
print(f"║  Colunas  : {len(tabela_final.columns):<41}║")
print(f"║  Anos     : {str(anos_ordenados):<41}║")
print("╚══════════════════════════════════════════════════════╝")



⚙️  Adicionando código IBGE...
✅ Todos os municípios tiveram código IBGE encontrado.
⚙️  Adicionando Regional de Saúde...
✅ Todos os municípios tiveram Regional de Saúde encontrada.

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo  : tabela_populacao_faixa_etaria.csv        ║
║  Linhas   : 853                                      ║
║  Colunas  : 57                                       ║
║  Anos     : [1970, 1980, 1991, 2000, 2010, 2022]     ║
╚══════════════════════════════════════════════════════╝


---
## 8. Inspeção Pós-Processamento

Verificação da tabela final antes de entregar: shape, amostra, totais e valores ausentes.

In [11]:
display(Markdown(f"**Shape final:** {tabela_final.shape[0]} linhas × {tabela_final.shape[1]} colunas"))

# Preview: colunas fixas + totais por ano
cols_preview = ["MUNICÍPIO", "CÓDIGO IBGE", "REGIONAL DE SAÚDE"] + [f"{a}_TOTAL" for a in anos_ordenados]
display(Markdown("**Amostra (colunas de total por ano):**"))
display(tabela_final[cols_preview].head(5))

# Estatísticas dos totais
total_pop = {ano: int(tabela_final[f"{ano}_TOTAL"].sum()) for ano in anos_ordenados}
display(Markdown("**População total por ano (soma de todos os municípios):**"))
display(
    pd.DataFrame({"Ano": list(total_pop.keys()), "População Total": list(total_pop.values())})
    .style
    .format({"População Total": lambda x: f"{x:,.0f}".replace(",", ".")})
    .hide(axis="index")
)

# Municípios sem código IBGE
sem_cod = tabela_final["CÓDIGO IBGE"].isna().sum() if "CÓDIGO IBGE" in tabela_final.columns else 0
if sem_cod > 0:
    display(Markdown(f"⚠️ **{sem_cod} município(s) sem código IBGE.**"))
else:
    display(Markdown("✅ **Todos os municípios possuem código IBGE.**"))

# Municípios sem Regional de Saúde
sem_regional = tabela_final["REGIONAL DE SAÚDE"].isna().sum() if "REGIONAL DE SAÚDE" in tabela_final.columns else 0
if sem_regional > 0:
    display(Markdown(f"⚠️ **{sem_regional} município(s) sem Regional de Saúde.**"))
else:
    display(Markdown("✅ **Todos os municípios possuem Regional de Saúde.**"))

# Valores ausentes gerais
missing = tabela_final.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    display(Markdown(f"**Colunas com valores ausentes:** {len(missing)}"))
    display(missing.to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes.**"))

**Shape final:** 853 linhas × 57 colunas

**Amostra (colunas de total por ano):**

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_TOTAL,1980_TOTAL,1991_TOTAL,2000_TOTAL,2010_TOTAL,2022_TOTAL
0,abadia dos dourados,3100104,uberlandia,4848,3936,3252,3105,3676,3111
1,abaete,3100203,sete lagoas,10206,9926,10560,10947,11171,11474
2,abre campo,3100302,manhumirim,9779,9375,9131,6610,7238,6903
3,acaiaca,3100401,ponte nova,1699,1822,1937,1875,1881,2012
4,aguanil,3100807,divinopolis,1312,1560,1576,1753,1938,2196


**População total por ano (soma de todos os municípios):**

Ano,População Total
1970,5.772.606
1980,7.339.191
1991,8.346.023
2000,9.255.322
2010,9.236.906
2022,10.524.280


✅ **Todos os municípios possuem código IBGE.**

✅ **Todos os municípios possuem Regional de Saúde.**

✅ **Sem valores ausentes.**

---
## 9. Exportação do Arquivo CSV Final

Gera o arquivo CSV final com os dados tratados de população por faixa etária, já com código IBGE, totais por ano e sem valores ausentes.

In [12]:
SAIDA.parent.mkdir(parents=True, exist_ok=True)
tabela_final.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("╔══════════════════════════════════════════════════════╗")
print("║           CSV final exportado com sucesso             ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Caminho  : {str(SAIDA):<41}║")
print(f"║  Linhas   : {len(tabela_final):<41}║")
print(f"║  Colunas  : {len(tabela_final.columns):<41}║")
print("╚══════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║           CSV final exportado com sucesso             ║
╠══════════════════════════════════════════════════════╣
║  Caminho  : /home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv║
║  Linhas   : 853                                      ║
║  Colunas  : 57                                       ║
╚══════════════════════════════════════════════════════╝
